# Real-Time Object Detection: PyTorch vs. NVIDIA TensorRT

This notebook takes a pretrained **YOLO11 nano** object detector and makes it faster with **NVIDIA TensorRT**, then proves the speedup three ways:

1. **Speed**: frames per second (FPS) on the same video, measured the same way for every version.
2. **Accuracy**: mAP on the coco128 dataset, so we can see what (if anything) we gave up for the speed.
3. **Visual demo**: a side-by-side video with live FPS counters.

**Versions compared**

| Version | What it is |
|---|---|
| PyTorch FP32 | The original model, 32-bit floating point numbers. This is the baseline. |
| TensorRT FP16 | Same model compiled by TensorRT, using 16-bit numbers. |
| TensorRT INT8 | Same model compiled by TensorRT, using 8-bit integers (optional). |

**Before you start:** Runtime > Change runtime type > **T4 GPU**, upload `video.mp4`, then Runtime > Run all.

## 1. Setup

### 1.1 Check that a GPU is available

TensorRT only runs on NVIDIA GPUs, so the first thing we do is ask the machine whether it has one. `nvidia-smi` is NVIDIA's tool that lists the GPUs, their memory, and the driver version.

If there is no GPU, the cell stops the notebook with a message telling you how to switch Colab to a T4 GPU. Failing early with a clear message is better than failing 10 minutes later with a confusing error.

In [ ]:
import shutil
import subprocess

gpu_found = (
    shutil.which('nvidia-smi') is not None
    and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
)

if not gpu_found:
    raise RuntimeError(
        'No NVIDIA GPU found. In Colab, go to Runtime > Change runtime type, '
        'select T4 GPU, click Save, then run the notebook again from the top.'
    )

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

### 1.2 Install the libraries

- **ultralytics**: the library that provides the YOLO models and a one-line export to TensorRT.
- **onnx / onnxslim**: the export goes PyTorch > ONNX (a standard model file format) > TensorRT engine. These handle the middle step.
- **tensorrt**: NVIDIA's inference optimizer. We install the build that matches the CUDA version PyTorch was built with (CUDA 12 on Colab), so the two libraries agree on which GPU runtime to use.

This takes a minute or two.

In [ ]:
import torch

cuda_major = torch.version.cuda.split('.')[0]  # '12' on Colab today
print(f'PyTorch was built for CUDA {torch.version.cuda}, installing tensorrt-cu{cuda_major}')

!pip install -q ultralytics onnx onnxslim tensorrt-cu{cuda_major}

### 1.3 Record the software and hardware versions

Benchmark numbers only mean something if you know what they were measured on. This cell prints the versions of Python, PyTorch, CUDA, TensorRT, and the GPU name, and saves them to `environment.txt` so you can copy them into the README.

In [ ]:
import platform
import subprocess

import cv2
import tensorrt
import torch
import ultralytics

driver = subprocess.run(
    ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
    capture_output=True, text=True,
).stdout.strip()

GPU_NAME = torch.cuda.get_device_name(0)

env = {
    'GPU': GPU_NAME,
    'NVIDIA driver': driver,
    'Python': platform.python_version(),
    'PyTorch': torch.__version__,
    'CUDA (PyTorch build)': torch.version.cuda,
    'TensorRT': tensorrt.__version__,
    'Ultralytics': ultralytics.__version__,
    'OpenCV': cv2.__version__,
}

width = max(len(k) for k in env)
with open('environment.txt', 'w') as f:
    for key, value in env.items():
        line = f'{key:<{width}} : {value}'
        print(line)
        print(line, file=f)

### 1.4 Settings used by every experiment

Keeping all the settings in one place guarantees every version is tested under identical conditions: same model, same image size, same number of warm-up frames, same video.

In [ ]:
from pathlib import Path

MODEL_NAME = 'yolo11n.pt'      # YOLO11 nano: the smallest, fastest pretrained YOLO11 model
IMG_SIZE = 640                 # every frame is fed to the model as a 640x640 image
WARMUP_FRAMES = 10             # frames run but not timed (see section 3)
VIDEO_PATH = Path('video.mp4')
EVAL_DATA = 'coco128.yaml'     # 128 COCO images, downloaded automatically by ultralytics

STEM = Path(MODEL_NAME).stem
FP16_ENGINE = f'{STEM}_fp16.engine'
INT8_ENGINE = f'{STEM}_int8.engine'

## 2. Input video

The benchmark runs on your own video, which must be named exactly `video.mp4` and uploaded to the default Colab folder (`/content`).

**How to upload:** click the **folder icon** in Colab's left sidebar, click the **upload icon** (a page with an up arrow), and pick your file. Wait for the upload to finish before running this cell.

This cell checks the file exists and prints its resolution, frame rate, and length.

In [ ]:
import cv2

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        'video.mp4 was not found. Click the folder icon in the left sidebar, '
        'then the upload icon (page with an up arrow), and upload your video. '
        'Name it exactly video.mp4 and leave it in /content (the folder that opens by default), '
        'then run this cell again.'
    )

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError('video.mp4 exists but OpenCV cannot open it. Try re-exporting it as a standard H.264 MP4.')

SRC_FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'Resolution : {src_w}x{src_h}')
print(f'Frame rate : {SRC_FPS:.1f} FPS')
print(f'Frames     : {n_frames} (~{n_frames / SRC_FPS:.1f} s)')

if n_frames < WARMUP_FRAMES + 50:
    print('Warning: this video is very short. Use at least ~10 seconds for stable FPS numbers.')

## 3. Baseline: PyTorch

### 3.1 Helper functions for a fair benchmark

Three details make the measurement fair:

- **Same input for everyone.** Each frame is resized (keeping its shape) and padded to a 640x640 square *before* the timer starts. The TensorRT engine is built for exactly 640x640, so we give PyTorch the exact same image. Otherwise PyTorch might quietly process a smaller rectangle and the comparison would be unfair.
- **GPU-synchronized timing.** The GPU works asynchronously: Python can hand it work and move on before the work is done. `torch.cuda.synchronize()` waits until the GPU is truly finished, so the stopwatch measures real GPU time, not just the time to *queue* the work.
- **Warm-up frames are skipped.** The first few frames are slow because the GPU is loading code, allocating memory, and raising its clock speed. Real applications run for a long time, so we skip the first 10 frames and time the rest.

We record two numbers:
- **FPS (end-to-end):** full model call per frame, including pre-processing, the neural network, and post-processing (removing duplicate boxes). Video decoding is not included.
- **Model inference ms:** time spent in the neural network alone, as reported by ultralytics. This is the part TensorRT actually speeds up.

In [ ]:
import time

import numpy as np
import torch
from ultralytics import YOLO


def letterbox(frame, size=IMG_SIZE, color=(114, 114, 114)):
    # Resize keeping the aspect ratio, then pad with gray to a size x size square.
    # Returns the square image and where the real picture sits inside it.
    h, w = frame.shape[:2]
    scale = size / max(h, w)
    nh, nw = round(h * scale), round(w * scale)
    resized = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas = np.full((size, size, 3), color, dtype=np.uint8)
    canvas[top:top + nh, left:left + nw] = resized
    return canvas, (top, left, nh, nw)


def read_frames(path):
    cap = cv2.VideoCapture(str(path))
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        yield frame
    cap.release()


def timed_predict(model, image):
    # Run one frame and return (result, seconds), with the GPU fully finished on both ends.
    torch.cuda.synchronize()
    start = time.perf_counter()
    result = model.predict(image, imgsz=IMG_SIZE, device=0, verbose=False)[0]
    torch.cuda.synchronize()
    return result, time.perf_counter() - start


def benchmark(model, name):
    frame_times, inference_ms = [], []
    for i, frame in enumerate(read_frames(VIDEO_PATH)):
        image, _ = letterbox(frame)
        result, seconds = timed_predict(model, image)
        if i < WARMUP_FRAMES:
            continue
        frame_times.append(seconds)
        inference_ms.append(result.speed['inference'])

    if not frame_times:
        raise RuntimeError(f'Video has {WARMUP_FRAMES} frames or fewer; upload a longer video.')

    stats = {
        'frames_timed': len(frame_times),
        'fps': len(frame_times) / sum(frame_times),
        'latency_ms': 1000 * float(np.mean(frame_times)),
        'inference_ms': float(np.mean(inference_ms)),
    }
    fps, latency, infer = stats['fps'], stats['latency_ms'], stats['inference_ms']
    print(f'{name}: {fps:.1f} FPS | {latency:.2f} ms per frame | {infer:.2f} ms model inference '
          f'({len(frame_times)} frames timed)')
    return stats


bench = {}  # benchmark results for every version, filled in below

### 3.2 Run the PyTorch baseline

This loads the pretrained YOLO11 nano model (the weights download automatically, about 5 MB) and runs it on every frame of the video in normal PyTorch, using 32-bit floating point (FP32) math. This is the number every other version is compared against.

In [ ]:
pt_model = YOLO(MODEL_NAME)
bench['PyTorch FP32'] = benchmark(pt_model, 'PyTorch FP32')

## 4. TensorRT FP16

Now we **export** the same model to a TensorRT **engine**. During the export TensorRT:

- **fuses layers**: merges steps like convolution + activation into one GPU operation, so data makes fewer trips to memory;
- **tests many GPU kernel implementations** for each layer and keeps the fastest one *for this exact GPU*;
- **uses FP16 (half precision)**: 16-bit numbers instead of 32-bit. They take half the memory and run on the T4's **Tensor Cores**, special hardware built for fast matrix math.

The result is a `.engine` file tuned for this GPU. Building it takes a few minutes; that is a one-time cost. We then run the exact same video with the exact same settings and timing code.

In [ ]:
import shutil

exported = YOLO(MODEL_NAME).export(format='engine', half=True, imgsz=IMG_SIZE, device=0)
shutil.move(exported, FP16_ENGINE)  # rename so the INT8 export does not overwrite it
print('Saved', FP16_ENGINE)

trt_fp16 = YOLO(FP16_ENGINE, task='detect')
bench['TensorRT FP16'] = benchmark(trt_fp16, 'TensorRT FP16')

## 5. TensorRT INT8 (optional)

INT8 goes one step further and stores numbers as 8-bit integers. That is faster still, but 8 bits can only represent 256 values, so TensorRT needs to learn the typical range of values flowing through each layer. This is called **calibration**: we run a small set of real images (coco128) through the model and TensorRT picks a scale for each layer.

INT8 export is more fragile than FP16, so it is wrapped in `try/except`. If it fails, the notebook prints a note and carries on with just PyTorch and FP16.

In [ ]:
trt_int8 = None
try:
    exported = YOLO(MODEL_NAME).export(
        format='engine', int8=True, data=EVAL_DATA, imgsz=IMG_SIZE, device=0
    )
    shutil.move(exported, INT8_ENGINE)
    print('Saved', INT8_ENGINE)
    trt_int8 = YOLO(INT8_ENGINE, task='detect')
    bench['TensorRT INT8'] = benchmark(trt_int8, 'TensorRT INT8')
except Exception as e:
    trt_int8 = None
    bench.pop('TensorRT INT8', None)
    print('NOTE: INT8 export or benchmark failed, so INT8 is skipped. '
          'The rest of the notebook continues with PyTorch and TensorRT FP16.')
    print('Reason:', repr(e))

## 6. Accuracy check

Speed is only useful if the model still finds objects correctly. We run each version on the coco128 validation set (128 labeled images) and record:

- **mAP50**: how well predicted boxes match the true boxes, counting a box as correct if it overlaps the true box by at least 50%.
- **mAP50-95**: the same score averaged over stricter overlap thresholds from 50% to 95%. This is the main COCO metric and rewards precisely placed boxes.

Both range from 0 to 1; higher is better. What matters here is the *difference* between versions. (coco128 images come from COCO's training set, so the absolute scores are higher than on the official validation set.)

In [ ]:
weights = {
    'PyTorch FP32': MODEL_NAME,
    'TensorRT FP16': FP16_ENGINE,
    'TensorRT INT8': INT8_ENGINE,
}

for name in bench:
    try:
        metrics = YOLO(weights[name], task='detect').val(
            data=EVAL_DATA, imgsz=IMG_SIZE, batch=1, device=0, plots=False, verbose=False
        )
        bench[name]['map50_95'] = float(metrics.box.map)
        bench[name]['map50'] = float(metrics.box.map50)
    except Exception as e:
        print(f'Validation failed for {name}: {e!r}')
        bench[name]['map50_95'] = float('nan')
        bench[name]['map50'] = float('nan')

for name, r in bench.items():
    print(f"{name:<14} mAP50-95 = {r['map50_95']:.3f}   mAP50 = {r['map50']:.3f}")

## 7. Results

### 7.1 Results table

One row per version: FPS, speedup compared to the PyTorch baseline, and accuracy. The table is saved as `results.csv` (for spreadsheets) and `results.md` (ready to paste into the GitHub README).

In [ ]:
import math

import pandas as pd

base_fps = bench['PyTorch FP32']['fps']


def fmt_map(value):
    return 'n/a' if math.isnan(value) else f'{value:.3f}'


rows = []
for name, r in bench.items():
    rows.append({
        'Version': name,
        'FPS': f"{r['fps']:.1f}",
        'Speedup vs. PyTorch': f"{r['fps'] / base_fps:.2f}x",
        'Model inference (ms)': f"{r['inference_ms']:.2f}",
        'mAP50-95': fmt_map(r['map50_95']),
        'mAP50': fmt_map(r['map50']),
    })
df = pd.DataFrame(rows)
df.to_csv('results.csv', index=False)

header = '| ' + ' | '.join(df.columns) + ' |'
divider = '|' + '|'.join(['---'] * len(df.columns)) + '|'
body = ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.itertuples(index=False)]
notes = [
    '',
    f'GPU: {GPU_NAME}. Model: {STEM}. Input: {IMG_SIZE}x{IMG_SIZE}, batch size 1. ',
    f'FPS is end-to-end per frame (pre-processing + inference + post-processing, excluding video decode), '
    f'averaged over the video after {WARMUP_FRAMES} warm-up frames. mAP measured on coco128.',
]
with open('results.md', 'w') as f:
    f.write('\n'.join([header, divider] + body + notes) + '\n')

display(df)

### 7.2 FPS bar chart

A picture is quicker to read than a table. Each bar is one version, labeled with its FPS and its speedup over PyTorch. Saved as `fps_chart.png` for the README.

In [ ]:
import matplotlib.pyplot as plt

names = list(bench)
fps_values = [bench[n]['fps'] for n in names]
speedups = [f / base_fps for f in fps_values]
colors = ['#8C8C8C' if n.startswith('PyTorch') else '#76B900' for n in names]  # gray baseline, NVIDIA green

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, fps_values, color=colors, width=0.6)
for bar, fps, speedup in zip(bars, fps_values, speedups):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{fps:.0f} FPS\n{speedup:.2f}x',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Frames per second (higher is better)')
ax.set_title(f'{STEM} on {GPU_NAME}, {IMG_SIZE}x{IMG_SIZE}, batch 1')
ax.set_ylim(0, max(fps_values) * 1.25)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('fps_chart.png', dpi=150)
plt.show()

## 8. Demo video

### 8.1 Side-by-side video

For each frame we run both the PyTorch model and the TensorRT FP16 engine, draw their detection boxes, and put the two results next to each other: **PyTorch on the left, TensorRT FP16 on the right**. Each side shows a live FPS counter (a rolling average over the last 30 frames), so the speed difference is visible at a glance.

OpenCV writes a raw video first; then `ffmpeg` re-encodes it to H.264, the format browsers and phones can play. To keep the run time reasonable, the demo uses at most the first 600 frames (about 20 seconds).

In [ ]:
from collections import deque

DEMO_MAX_FRAMES = 600
RAW_DEMO = 'demo_raw.mp4'


def draw_label(img, title, fps):
    s = img.shape[1] / 640  # scale text with the panel width
    cv2.rectangle(img, (0, 0), (int(270 * s), int(72 * s)), (0, 0, 0), -1)
    cv2.putText(img, title, (int(10 * s), int(30 * s)), cv2.FONT_HERSHEY_SIMPLEX,
                0.8 * s, (255, 255, 255), max(1, int(2 * s)), cv2.LINE_AA)
    cv2.putText(img, f'FPS: {fps:.1f}', (int(10 * s), int(62 * s)), cv2.FONT_HERSHEY_SIMPLEX,
                0.8 * s, (0, 185, 118), max(1, int(2 * s)), cv2.LINE_AA)


pt_times, trt_times = deque(maxlen=30), deque(maxlen=30)
writer = None

for i, frame in enumerate(read_frames(VIDEO_PATH)):
    if i >= DEMO_MAX_FRAMES:
        break
    image, (top, left, nh, nw) = letterbox(frame)

    pt_result, pt_seconds = timed_predict(pt_model, image)
    trt_result, trt_seconds = timed_predict(trt_fp16, image)
    pt_times.append(pt_seconds)
    trt_times.append(trt_seconds)

    # Draw boxes, then crop away the gray padding so only the real picture remains.
    left_panel = pt_result.plot()[top:top + nh, left:left + nw].copy()
    right_panel = trt_result.plot()[top:top + nh, left:left + nw].copy()
    draw_label(left_panel, 'PyTorch', len(pt_times) / sum(pt_times))
    draw_label(right_panel, 'TensorRT FP16', len(trt_times) / sum(trt_times))

    separator = np.full((nh, 4, 3), 255, dtype=np.uint8)
    combined = np.hstack([left_panel, separator, right_panel])

    if writer is None:
        size = (combined.shape[1], combined.shape[0])
        writer = cv2.VideoWriter(RAW_DEMO, cv2.VideoWriter_fourcc(*'mp4v'), SRC_FPS, size)
    writer.write(combined)

writer.release()

subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error', '-i', RAW_DEMO,
    '-vf', 'scale=trunc(iw/2)*2:trunc(ih/2)*2',  # H.264 needs even width and height
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '23', '-movflags', '+faststart',
    'demo_side_by_side.mp4',
], check=True)
print('Saved demo_side_by_side.mp4')

### 8.2 GIF for the README

GitHub READMEs can show GIFs inline but not MP4s, so we make a short GIF (first 8 seconds). GIFs get large quickly, so we lower the frame rate and size and use a 128-color palette. If the result is over 10 MB, the cell automatically tries smaller settings.

In [ ]:
import os

from IPython.display import Image

GIF_SECONDS = 8

for gif_width, gif_fps in [(800, 12), (640, 10), (480, 8), (360, 6)]:
    palette_filter = (
        f'fps={gif_fps},scale={gif_width}:-1:flags=lanczos,split[a][b];'
        '[a]palettegen=max_colors=128[p];[b][p]paletteuse=dither=bayer:bayer_scale=5'
    )
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error', '-t', str(GIF_SECONDS),
        '-i', 'demo_side_by_side.mp4', '-vf', palette_filter, '-loop', '0', 'demo.gif',
    ], check=True)
    size_mb = os.path.getsize('demo.gif') / 1e6
    print(f'demo.gif at {gif_width}px wide, {gif_fps} fps: {size_mb:.1f} MB')
    if size_mb < 10:
        break

display(Image('demo.gif'))

## 9. Download the results

Colab deletes every file when the session ends, so this cell bundles the results into `results.zip` and downloads it to your computer. (`environment.txt` is included too, with the versions for the README.)

If the download does not start, open the folder panel on the left, right-click `results.zip`, and choose **Download**.

In [ ]:
import zipfile

outputs = ['results.csv', 'results.md', 'fps_chart.png',
           'demo_side_by_side.mp4', 'demo.gif', 'environment.txt']

with zipfile.ZipFile('results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in outputs:
        if os.path.exists(name):
            zf.write(name)
        else:
            print('Missing, not zipped:', name)

print(f"results.zip: {os.path.getsize('results.zip') / 1e6:.1f} MB")

try:
    from google.colab import files
    files.download('results.zip')
except ImportError:
    print('Not running in Colab; results.zip is saved in the current folder.')